# Demo 1 — Prompt Anatomy Builder
## Session 5: Core Prompt Engineering Techniques

**Goal:** Watch output quality improve as each of the 5 prompt components is added to the same task.

**Task:** Bug triage — classify a bug report, assign severity, recommend owner team.

**What to watch:** After each cell, notice how the model's response changes. The same model, the same bug — only the prompt structure changes.

**Expected outcome:** By v5, the model returns a clean, structured JSON triage with correct severity and team. Each component contributes a specific, measurable improvement.

---
**Components added step by step:**
1. Bare task only — baseline
2. + Role / PCT pattern (~37% relevance improvement)
3. + Context (positional bias: context at top, query at end → +30% quality)
4. + Constraints + JSON output schema
5. + XML structural tags (injection defence + parsing clarity)
6. Side-by-side comparison table

In [1]:
# -- Setup -----------------------------------------------------------------
import os, textwrap, pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display

load_dotenv()

client = OpenAI(base_url=os.environ["API_BASE_URL"], api_key=os.environ["API_KEY"])
MODEL = os.environ.get("MODEL_NAME", "claude-sonnet-4-6")

# -- Shared input across ALL cells (never changes) -------------------------
BUG_REPORT = """Title: Payment confirmation email not sent after checkout
Description: Users complete purchase but receive no confirmation email.
Steps: Add item to cart -> checkout -> pay with Stripe -> order shows 'confirmed'
  in dashboard but no email arrives.
Frequency: ~40% of transactions in the past 24 hours.
Environment: Production. Stripe webhooks show 'charge.succeeded'.
Impact: Customer support receiving 80+ tickets/hour."""

TEAM_CONTEXT = """Engineering teams:
- Payments: Stripe integration, billing, refunds
- Notifications: email/SMS/push, SendGrid integration, templates
- Platform: infrastructure, queues, background jobs
- Customer Experience: support tooling

Severity model:
- P0: Revenue impact or data loss, >20% users affected
- P1: Major feature broken, 5-20% affected, no workaround
- P2: Feature degraded, <5% affected or workaround exists
- P3: Cosmetic / low impact"""

results = {}

def show_prompt(system, user, label=""):
    w = 68
    tag = f" {label} " if label else " PROMPT "
    print(tag.center(w, "="))
    print("[ SYSTEM ]")
    for line in textwrap.wrap(system.strip(), width=w - 2):
        print(f"  {line}")
    print()
    print("[ USER ]")
    for line in user.strip().split("\n"):
        print(f"  {line}")
    print("=" * w)
    print()

def ask(system, user, label, json_mode=False):
    show_prompt(system, user, label)
    kwargs = dict(model=MODEL, max_tokens=512,
                  messages=[{"role": "system", "content": system},
                             {"role": "user",   "content": user}])
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    resp = client.chat.completions.create(**kwargs)
    text = resp.choices[0].message.content
    results[label] = text
    return text

print(f"Model: {MODEL}")
print("Bug report loaded. All 5 cells use this exact input.")

Model: llama3.2:1b
Bug report loaded. All 5 cells use this exact input.


## Cell 1 — Bare Task
**What we're adding:** Nothing. Just the raw task instruction.
**What to expect:** The model responds, but without structure, severity, or team assignment.

In [2]:
# COMPONENT 1: Task only
# Why: This is our baseline — what you get with zero prompt engineering.

output_v1 = ask(
    system="You are a helpful assistant.",
    user=f"Triage this bug report:\n\n{BUG_REPORT}",
    label="v1 - Bare Task"
)
print("-- RESPONSE --")
print(output_v1)

========================== v1 - Bare Task ==========================
[ SYSTEM ]
  You are a helpful assistant.

[ USER ]
  Triage this bug report:
  
  Title: Payment confirmation email not sent after checkout
  Description: Users complete purchase but receive no confirmation email.
  Steps: Add item to cart -> checkout -> pay with Stripe -> order shows 'confirmed'
    in dashboard but no email arrives.
  Frequency: ~40% of transactions in the past 24 hours.
  Environment: Production. Stripe webhooks show 'charge.succeeded'.
  Impact: Customer support receiving 80+ tickets/hour.



-- RESPONSE --
I'd be happy to help you triage this bug report. Here's a breakdown of the key points:

**Issue Description**: Users who complete purchases are not receiving payment confirmation emails.

**Key Metrics**:

* Frequency: approximately 40% of transactions in the past 24 hours.
* Environment: production environment with Stripe webhooks showing 'charge.succeeded'.

**Steps and Analysis**:

1. **Checkout process**: The user adds an item to their cart, proceeds to checkout using Stripe, and completes the payment.
2. **Stripe webhook**: Payment processing via Stripe should trigger a webhook to notify you of the charge success.
3. **No confirmation email**: Despite receiving no actual confirmation, users still receive emails as they complete purchases from your dashboard.

**Analysis**:

* The 40% frequency rate suggests that this issue is intermittent and may be related to various factors such as:
	+ Browser or device compatibility issues
	+ Network connectivity problems
	+ Serv

## Cell 2 — + Role (PCT Pattern)
**What we're adding:** A Persona-Context-Task role in the system prompt.
**Why:** The PCT pattern grounds the model's vocabulary and expertise level (~37% improvement).
**What to expect:** More technical language, domain-appropriate framing.

In [3]:
# COMPONENT 2: + Role / PCT pattern
# Why: Role tells the model HOW to think, not just what to do.

output_v2 = ask(
    system="""You are a senior software engineer on a platform team.
You have 8 years of experience triaging production incidents across
e-commerce, payments, and notification systems.""",
    user=f"Triage this bug report:\n\n{BUG_REPORT}",
    label="v2 - + Role/PCT"
)
print("-- RESPONSE --")
print(output_v2)

========================= v2 - + Role/PCT ==========================
[ SYSTEM ]
  You are a senior software engineer on a platform team. You have 8
  years of experience triaging production incidents across
  e-commerce, payments, and notification systems.

[ USER ]
  Triage this bug report:
  
  Title: Payment confirmation email not sent after checkout
  Description: Users complete purchase but receive no confirmation email.
  Steps: Add item to cart -> checkout -> pay with Stripe -> order shows 'confirmed'
    in dashboard but no email arrives.
  Frequency: ~40% of transactions in the past 24 hours.
  Environment: Production. Stripe webhooks show 'charge.succeeded'.
  Impact: Customer support receiving 80+ tickets/hour.



-- RESPONSE --
I'm ready to help you triage this bug report. Here's a breakdown of the issue and potential solutions:

**Issue Statement:**

* The payment confirmation email is not being sent after completing a purchase, despite receiving an order status of "confirmed" in the dashboard.
* Despite successful Stripe webhooks indicating that a charge has been successfully processed, the customer support team is receiving a high volume of ticket inquiries related to missing emails.

**Key Findings:**

1. The email was never sent due to incomplete delivery settings or misconfiguration.
2. Stripe webhooks are indicating successful charge processing, but no payment confirmation email is being sent.
3. High customer support ticket volume suggests that the issue persists across multiple transactions.

**Potential Solutions:**

1. **Verify and Optimize Delivery Settings**: Ensure that the email sending settings are correct for the production environment, including:
	* Mailchimp or similar integr

## Cell 3 — + Context
**What we're adding:** Team structure and severity model.
**Why:** Without context, the model invents its own severity scale and team names.
**Positional bias rule:** Long context goes at the TOP, the specific query at the END (+30% quality).

In [4]:
# COMPONENT 3: + Context (at top, query at end — positional bias rule)
# Why: The model attends more strongly to recent tokens. Query at end = query is fresh.

output_v3 = ask(
    system="""You are a senior software engineer on a platform team.
You have 8 years of experience triaging production incidents across
e-commerce, payments, and notification systems.""",
    user=f"""Context about our engineering organisation:
{TEAM_CONTEXT}

Bug report to triage:
{BUG_REPORT}""",
    label="v3 - + Context"
)
print("-- RESPONSE --")
print(output_v3)

========================== v3 - + Context ==========================
[ SYSTEM ]
  You are a senior software engineer on a platform team. You have 8
  years of experience triaging production incidents across
  e-commerce, payments, and notification systems.

[ USER ]
  Context about our engineering organisation:
  Engineering teams:
  - Payments: Stripe integration, billing, refunds
  - Notifications: email/SMS/push, SendGrid integration, templates
  - Platform: infrastructure, queues, background jobs
  - Customer Experience: support tooling
  
  Severity model:
  - P0: Revenue impact or data loss, >20% users affected
  - P1: Major feature broken, 5-20% affected, no workaround
  - P2: Feature degraded, <5% affected or workaround exists
  - P3: Cosmetic / low impact
  
  Bug report to triage:
  Title: Payment confirmation email not sent after checkout
  Description: Users complete purchase but receive no confirmation email.
  Steps: Add item to cart -> checkout -> pay with Stripe -> orde

-- RESPONSE --
Based on the context and bug report, I'll provide a triaged ticket to prioritize the issue:

**Ticket Triage Report**

**Title:** Payment confirmation email not sent after checkout (P0)

**Description:**

The payment confirmation email is not being sent to users who complete their purchases through Stripe, despite receiving revenue. This results in an unsustainable pressure on our customer support team.

**Steps:**

1. Add item to cart
2. Checkout -> pay with Stripe
3. Order shows 'confirmed'
4. In the dashboard, no email arrives

**Frequency:** 40% of transactions in the past 24 hours

**Environment:** Production, Stripe webhooks show 'charge.succeeded'

**Impact:**

* Customer support receiving 80+ tickets/hour due to missing emails
* Productivity and sanity of our engineering teams suffering from constant bug reports

**Recommendation:**

Prioritize this ticket by:

1. Adding a flag to the payment confirmation pipeline to indicate that Stripe completed a transaction b

## Cell 4 — + Constraints + Output Format
**What we're adding:** Explicit rules (positive framing) and a JSON output schema.
**Why:** Without a format spec, the model chooses its own structure — which changes every call.
**Note:** Watch how the JSON schema appears in the prompt below.

In [5]:
# COMPONENT 4: + Constraints + Output Format
# Why: Specifying the output schema makes the response machine-readable and consistent.
# NOTE: system is a plain string (not f-string) so { } are literal — no escaping needed.

JSON_SCHEMA = """{
  "severity": "P0 or P1 or P2 or P3",
  "owner_team": "Payments or Notifications or Platform or Customer Experience",
  "root_cause_hypothesis": "<one sentence>",
  "first_step": "<concrete investigation action>",
  "confidence": "high or medium or low"
}"""

output_v4 = ask(
    system=f"""You are a senior software engineer on a platform team.
You have 8 years of experience triaging production incidents.

When triaging, always:
- Identify the most likely root cause based on evidence in the report
- Assign severity using the P0-P3 model
- Recommend exactly one owning team from: Payments, Notifications, Platform, Customer Experience
- Suggest the single most impactful first investigation step

Respond ONLY with valid JSON matching this exact schema:
{JSON_SCHEMA}""",
    user=f"""Context about our engineering organisation:
{TEAM_CONTEXT}

Bug report to triage:
{BUG_REPORT}""",
    label="v4 - + Constraints + Format",
    json_mode=True
)
print("-- RESPONSE --")
print(output_v4)

=================== v4 - + Constraints + Format ====================
[ SYSTEM ]
  You are a senior software engineer on a platform team. You have 8
  years of experience triaging production incidents.  When triaging,
  always: - Identify the most likely root cause based on evidence in
  the report - Assign severity using the P0-P3 model - Recommend
  exactly one owning team from: Payments, Notifications, Platform,
  Customer Experience - Suggest the single most impactful first
  investigation step  Respond ONLY with valid JSON matching this
  exact schema: {   "severity": "P0 or P1 or P2 or P3",
  "owner_team": "Payments or Notifications or Platform or Customer
  Experience",   "root_cause_hypothesis": "<one sentence>",
  "first_step": "<concrete investigation action>",   "confidence":
  "high or medium or low" }

[ USER ]
  Context about our engineering organisation:
  Engineering teams:
  - Payments: Stripe integration, billing, refunds
  - Notifications: email/SMS/push, SendGrid int

-- RESPONSE --
{
  "severity": "P0",
  "owner_team": "Customer Experience",
  "root_cause_hypothesis": "Stripe webhook not triggering email sending",
  "first_step": "Verify Stripe webhook is properly configured and set up correctly for notifications",
  "confidence": "high"
}


## Cell 5 — + XML Tags (Production-Ready)
**What we're adding:** Anthropic-style XML structural tags wrapping each section.
**Why two reasons:**
1. **Structural clarity** — the model can unambiguously distinguish instructions from data
2. **Injection defence** — content inside `<input>` is treated as data, not instructions

**This is the prompt you'd ship to production.**

In [6]:
# COMPONENT 5: + XML structural tags (production-ready)
# Why: XML tags create parsing boundaries — prevents data from being interpreted as instructions.

output_v5 = ask(
    system="""You are a senior software engineer on a platform team.
You have 8 years of experience triaging production incidents.""",
    user=f"""<instructions>
Triage the bug report in <input> using the context in <context>.
Assign severity (P0-P3), one owning team, root cause hypothesis, and first investigation step.
Respond ONLY with valid JSON:
{{
  "severity": "P0 or P1 or P2 or P3",
  "owner_team": "Payments or Notifications or Platform or Customer Experience",
  "root_cause_hypothesis": "<one sentence>",
  "first_step": "<concrete action>",
  "confidence": "high or medium or low"
}}
</instructions>

<context>
{TEAM_CONTEXT}
</context>

<input>
{BUG_REPORT}
</input>""",
    label="v5 - + XML Tags",
    json_mode=True
)
print("-- RESPONSE --")
print(output_v5)

========================= v5 - + XML Tags ==========================
[ SYSTEM ]
  You are a senior software engineer on a platform team. You have 8
  years of experience triaging production incidents.

[ USER ]
  <instructions>
  Triage the bug report in <input> using the context in <context>.
  Assign severity (P0-P3), one owning team, root cause hypothesis, and first investigation step.
  Respond ONLY with valid JSON:
  {
    "severity": "P0 or P1 or P2 or P3",
    "owner_team": "Payments or Notifications or Platform or Customer Experience",
    "root_cause_hypothesis": "<one sentence>",
    "first_step": "<concrete action>",
    "confidence": "high or medium or low"
  }
  </instructions>
  
  <context>
  Engineering teams:
  - Payments: Stripe integration, billing, refunds
  - Notifications: email/SMS/push, SendGrid integration, templates
  - Platform: infrastructure, queues, background jobs
  - Customer Experience: support tooling
  
  Severity model:
  - P0: Revenue impact or data

-- RESPONSE --
{
  "severity": "P1",
  "owner_team": "Notifications",
  "root_cause_hypothesis": "The Payment confirmation email not sent after checkout may be due to incomplete payment processing or errors with the Stripe webhooks.",
  "first_step": "Check that the customer enters a valid credit card number and verifies their address during checkout.",
  "confidence": "medium"
}


## Cell 6 — Comparison Table
**What we're seeing:** All 5 prompt versions side by side.
**Key question for the audience:** At which version did the output become useful for automation?

In [7]:
def truncate(text, n=110):
    return textwrap.shorten(text.strip().replace("\n", " "), width=n, placeholder="...")

comparison = [
    {"Version": "v1 - Bare task",         "Component added": "Task only",                      "Output": truncate(results["v1 - Bare Task"])},
    {"Version": "v2 - + Role",            "Component added": "+ PCT persona",                  "Output": truncate(results["v2 - + Role/PCT"])},
    {"Version": "v3 - + Context",         "Component added": "+ Team + severity model",         "Output": truncate(results["v3 - + Context"])},
    {"Version": "v4 - + Constraints/Fmt","Component added": "+ Rules + JSON schema",           "Output": truncate(results["v4 - + Constraints + Format"])},
    {"Version": "v5 - + XML tags",        "Component added": "+ Structural XML separation",    "Output": truncate(results["v5 - + XML Tags"])},
]
pd.set_option("display.max_colwidth", 115)
display(pd.DataFrame(comparison))
print()
print("Discussion:")
print("  - At which version does the output become parseable by code?")
print("  - What did adding the Role change? What did it NOT change?")
print("  - Same model. Same bug. Only the prompt changed.")

,Version,Component added,Output
0,v1 - Bare task,Task only,I'd be happy to help you triage this bug report. Here's a breakdown of the key points: **Issue...
1,v2 - + Role,+ PCT persona,I'm ready to help you triage this bug report. Here's a breakdown of the issue and potential solutions:...
2,v3 - + Context,+ Team + severity model,"Based on the context and bug report, I'll provide a triaged ticket to prioritize the issue: **Ticket Triage..."
3,v4 - + Constraints/Fmt,+ Rules + JSON schema,"{ ""severity"": ""P0"", ""owner_team"": ""Customer Experience"", ""root_cause_hypothesis"": ""Stripe webhook not..."
4,v5 - + XML tags,+ Structural XML separation,"{ ""severity"": ""P1"", ""owner_team"": ""Notifications"", ""root_cause_hypothesis"": ""The Payment confirmation email..."



Discussion:
  - At which version does the output become parseable by code?
  - What did adding the Role change? What did it NOT change?
  - Same model. Same bug. Only the prompt changed.
